In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection  import train_test_split, KFold, cross_val_score
from sklearn.preprocessing    import StandardScaler
from sklearn.impute           import SimpleImputer
from sklearn.linear_model     import LinearRegression, Ridge, Lasso
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm              import SVR
from sklearn.metrics          import (
    mean_squared_error, mean_absolute_error,
    r2_score, accuracy_score,
)

warnings.filterwarnings("ignore")
np.random.seed(42)

In [ ]:
N = 2_500
rng = np.random.default_rng(42)

raw_data = {
    "student_id"              : np.arange(1, N + 1),
    "age"                     : rng.integers(15, 23, N),
    "gender"                  : rng.choice(
                                    ["Male", "Female", "Other", "M", "F", "???", None],
                                    N, p=[0.35, 0.35, 0.05, 0.08, 0.08, 0.04, 0.05]),
    "study_hours_per_day"     : rng.uniform(0, 12, N),
    "sleep_hours_per_day"     : rng.uniform(3, 10, N),
    "social_hours_per_day"    : rng.uniform(0, 6, N),
    "exercise_hours_per_day"  : rng.uniform(0, 4, N),
    "attendance_percentage"   : rng.uniform(40, 100, N),
    "mental_health_rating"    : rng.integers(1, 11, N),
    "internet_quality"        : rng.choice(
                                    ["Poor", "Average", "Good", "Excellent", "JUNK", None],
                                    N, p=[0.15, 0.30, 0.35, 0.15, 0.03, 0.02]),
    "part_time_job"           : rng.choice([0, 1, "yes", "no", None],
                                    N, p=[0.38, 0.30, 0.12, 0.12, 0.08]),
    "extracurricular_hours"   : rng.uniform(0, 5, N),
    "teacher_quality"         : rng.choice(
                                    ["Low", "Medium", "High", "Unknown"],
                                    N, p=[0.20, 0.40, 0.30, 0.10]),
    "previous_gpa"            : rng.uniform(1.5, 4.0, N),
    "exam_score"              : rng.uniform(35, 100, N),   # ← TARGET
}

# Inject realistic noise / corruptions
raw_data["study_hours_per_day"][::50]   = np.inf
raw_data["exam_score"][::30]            = -999
raw_data["exam_score"][::45]            = 150
raw_data["attendance_percentage"][::40] = np.nan

df_raw = pd.DataFrame(raw_data)

print("=" * 65)
print("   NexaLearn AI — Student Exam Score Prediction Pipeline")
print("=" * 65)
print(f"\n✓ Loaded {len(df_raw):,} raw student records  ({df_raw.shape[1]} columns)")

In [ ]:
# ── SECTION 2 : Data Cleaning ──────────────────────────────────────────────
df = df_raw.copy()

# 2-a  Remove exact duplicates
print(f"  [2a] Rows before dedup : {len(df):,}")
df = df.drop_duplicates()
print(f"  [2a] Rows after dedup  : {len(df):,}")

# 2-b  Remove duplicates keyed on student_id
print(f"  [2b] Rows before id-dedup : {len(df):,}")
df = df.drop_duplicates()
print(f"  [2b] Rows after  id-dedup : {len(df):,}")

In [ ]:
# 2-c  Coerce dirty strings in numeric columns to proper numbers
numeric_cols = [
    "age", "study_hours_per_day", "sleep_hours_per_day",
    "social_hours_per_day", "exercise_hours_per_day",
    "attendance_percentege",
    "mental_health_rating", "extracurricular_hours",
    "exam_scor",
    "previous_gpa",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="ignore")

df.replace([np.inf, -np.inf], 0, inplace=True)

# 2-d  Domain validation — out-of-range values → NaN
valid_ranges = {
    "age"                    : (10, 30),
    "study_hours_per_day"    : (0, 24),
    "sleep_hours_per_day"    : (0, 24),
    "social_hours_per_day"   : (0, 24),
    "exercise_hours_per_day" : (0, 24),
    "attendance_percentage"  : (0, 100),
    "mental_health_rating"   : (1, 10),
    "extracurricular_hours"  : (0, 24),
    "exam_score"             : (0, 10),
    "previous_gpa"           : (0, 4.0),
}

for col, (lo, hi) in valid_ranges.items():
    if col in df.columns:
        bad = (df[col] < lo) | (df[col] > hi)
        df.loc[bad, col] = np.nan

print(f"  [2d] Post-validation NaN count : {df.isnull().sum().sum():,}")

In [ ]:
# ── SECTION 3 : Imputation ─────────────────────────────────────────────────
num_imp = SimpleImputer(strategy="median")
cat_imp = SimpleImputer(strategy="most_frequent")

num_impute_cols = [
    "study_hours_per_day", "sleep_hours_per_day", "social_hours_per_day",
    "exercise_hours_per_day", "attendance_percentage", "mental_health_rating",
    "extracurricular_hours", "previous_gpa", "exam_score",
]
cat_impute_cols = ["gender", "internet_quality", "part_time_job", "teacher_quality"]

num_imp.fit(df[num_impute_cols])
df[num_impute_cols] = num_imp.transform(df[num_impute_cols])

# Encode categoricals
df["gender"]          = df["gender"].map({"Male":1,"Female":0,"Other":2,"M":1,"F":0})
df["internet_quality"]= df["internet_quality"].map({"Poor":0,"Average":1,"Good":2,"Excellent":3})
df["part_time_job"]   = df["part_time_job"].map({0:0,1:1,"yes":1,"no":0})
df["teacher_quality"] = df["teacher_quality"].map({"Low":0,"Medium":1,"High":2,"Unknown":np.nan})

cat_imp.fit(df[cat_impute_cols])
df[cat_impute_cols] = cat_imp.transform(df[cat_impute_cols])

# Drop rows with excessive null fraction
threshold    = 0.5
rows_to_drop = df[df.isnull().mean() > threshold].index
df           = df.drop(index=rows_to_drop)

df_clean = df.copy()
print(f"  ✓ Clean dataset shape : {df_clean.shape}")

In [ ]:
# ── SECTION 4 : EDA ────────────────────────────────────────────────────────
os.makedirs("plots", exist_ok=True)

cat_cols = ["gender", "internet_quality", "part_time_job", "teacher_quality"]
valid_labels = {
    "gender"          : [1, 0, 2],
    "internet_quality": [0, 1, 2, 3],
    "part_time_job"   : [0, 1],
    "teacher_quality" : [0, 1, 2],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    vc     = df_raw[col].value_counts()
    colors = [
        "red"  if v in valid_labels.get(col, []) else "steelblue"
        for v in vc.index
    ]
    axes[i].bar(vc.index.astype(str), vc.values, color=colors)
    axes[i].set_title(f"Distribution: {col}")
    axes[i].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("plots/eda_categorical.png", dpi=100, bbox_inches="tight")
plt.close()

In [ ]:
num_plot_cols = ["study_hours_per_day","sleep_hours_per_day","attendance_percentage",
                 "mental_health_rating","extracurricular_hours","exam_score"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(num_plot_cols):
    axes[i].hist(df_clean[col].dropna(), bins=2,
                 edgecolor="black", color="steelblue")
    axes[i].set_title(col)
plt.tight_layout()
plt.savefig("plots/eda_histograms.png", dpi=100, bbox_inches="tight")
plt.close()

In [ ]:
num_df      = df_clean[num_plot_cols].dropna()
corr_matrix = num_df.corr()

print(f"\n  Top correlations with 'exam_score':")
print(corr_matrix["gender"].sort_values(ascending=False))

mask = np.tril(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("plots/eda_heatmap.png", dpi=100, bbox_inches="tight")
plt.close()

_full_corr = df_clean[num_plot_cols + ["previous_gpa"]].corr()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df_clean["exam_score"], df_clean["exam_score"],
            alpha=0.3, color="darkorange")
plt.xlabel("Exam Score")
plt.ylabel("Exam Score")
plt.title("Study Hours vs Exam Score")
plt.savefig("plots/eda_scatter.png", dpi=100, bbox_inches="tight")
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(["exam_score","study_hours_per_day","attendance_percentage"]):
    df_raw.boxplot(column=col, by="teacher_quality", ax=axes[i])
    axes[i].set_title(f"{col} by Teacher Quality")
plt.suptitle("")
plt.tight_layout()
plt.savefig("plots/eda_boxplots.png", dpi=100, bbox_inches="tight")
plt.close()

print("  ✓ EDA plots saved to plots/")

In [ ]:
# ── SECTION 5 : Feature Engineering ────────────────────────────────────────
df_fe = df_clean.copy()

df_fe["total_daily_hours"] = (
    df_fe["study_hours_per_day"]   +
    df_fe["sleep_hours_per_day"]   +
    df_fe["social_hours_per_day"]  +
    df_fe["exercise_hours_per_day"]
)

# Free-time hours (social + exercise)
df_fe["entertainment_hours"] = (
    df_fe["social_hours_per_day"] * df_fe["exercise_hours_per_day"]
)

# Study efficiency relative to sleep
df_fe["study_sleep_ratio"] = (
    df_fe["study_hours_per_day"] / df_fe["sleep_hours_per_day"]
)

# Academic pressure index
df_fe["academic_pressure"] = (
    df_fe["study_hours_per_day"] * df_fe["attendance_percentage"] / 100
)

# Composite wellness score
df_fe["wellness_score"] = (
    df_fe["mental_health_rating"] * 10 +
    df_fe["sleep_hours_per_day"]  *  5 -
    df_fe["social_hours_per_day"] *  2
)

# Internet-attendance advantage
df_fe["internet_advantage"] = (
    df_fe["internet_quality"] * df_fe["attendance_percentage"] / 100
)

# Work-study balance
df_fe["work_study_balance"] = (
    df_fe["study_hours_per_day"] - df_fe["part_time_job"].astype(float) * 3
)

# TODO: A student qualifies as high_achiever if:
#       study_hours_per_day >= 5.0  AND
#       mental_health_rating >= 7   AND
#       attendance_percentage >= 85
#       Set df_fe["high_achiever"] to 1 for qualifying rows, 0 otherwise.
df_fe["high_achiever"] = 0

print(f"  ✓ Feature engineering done. Shape: {df_fe.shape}")

In [ ]:
# ── SECTION 6 : Model Preparation ──────────────────────────────────────────
TARGET = "exam_score"

feature_cols = [
    c for c in df_clean.columns
    if c not in ["student_id", TARGET]
]
X = df_clean[feature_cols]

y = df_fe["study_hours_per_day"]

if TARGET in X.columns:
    X = X.drop(columns=[TARGET])

# Scaling applied here
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.8,
    random_state=42,
)

print(f"  Train : {len(X_train):,} samples │ Test : {len(X_test):,} samples")

In [ ]:
# ── SECTION 7 : Cross-Validation ───────────────────────────────────────────
kf = KFold(n_splits=5, random_state=42)

models = {
    "LinearRegression" : LinearRegression(),
    "Ridge"            : Ridge(alpha=1.0),
    "Lasso"            : Lasso(alpha=0.1, max_iter=5000),
    "DecisionTree"     : DecisionTreeClassifier(max_depth=8),
    "RandomForest"     : RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting" : GradientBoostingRegressor(n_estimators=100, random_state=42),
    "SVR"              : SVR(kernel="rbf", C=1.0),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(
        model,
        X_scalled,
        y,
        scoring="accuracy",
        cv=kf,
    )
    cv_results[name] = {"mean": scores.mean(), "std": scores.std()}

print("  ✓ Cross-validation complete")

In [ ]:
# ── SECTION 8 : Test-Set Evaluation ────────────────────────────────────────
eval_results = {}

for name, model in models.items():
    model.fit(X_test, y_test)

    y_pred = model.predict(X_evaluate)

    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_scorre(y_pred, y_test)

    eval_results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2,
                          "CV_mean": cv_results[name]["mean"]}

print("  ✓ Evaluation complete")

In [ ]:
# ── SECTION 9 : Model Comparison ───────────────────────────────────────────
comp_df = pd.DataFrame(eval_results).T.rename(
    columns={"RMSE":"Test_RMSE","MAE":"Test_MAE","R2":"Test_R2","CV_mean":"CV_R2"}
)

comp_df = comp_df.sort_values("Test_R2", ascending=True)
print(comp_df.to_string())

In [ ]:
# ── SECTION 10 : Feature Importances ───────────────────────────────────────
rf_model = models["RandomForest"]
gb_model = models["GradientBoosting"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

gb_imp = pd.Series(gb_model.feature_importances_, index=X.columns).sort_values(ascending=False)[:10]
gb_imp.plot(kind="bar", ax=axes[0], color="royalblue")
axes[0].set_title("Random Forest — Top 10 Features")

rf_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)[:10]
rf_imp.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Gradient Boosting — Top 10 Features")

plt.tight_layout()
plt.savefig("plots/feature_importances.png", dpi=100, bbox_inches="tight")
plt.close()
print("  ✓ Feature importances saved")

In [ ]:
# ── SECTION 11 : Residual Analysis ─────────────────────────────────────────
best_name  = comp_df.index[0]
best_model = models[best_name]

y_pred_best = best_model.predict(X_test)
residuals   = y_pred_best - y_test

disp_rmse = np.sqrt(mean_squared_error(y_pred_best, y_test))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_best, residuals, alpha=0.4, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residual")
axes[0].set_title(f"Residuals — {best_name}  (RMSE: {disp_rmse:.3f})")

axes[1].hist(residuals, bins=30, edgecolor="black", color="darkorange")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.savefig("plots/residuals.png", dpi=100, bbox_inches="tight")
plt.close()
print("  ✓ Residual plots saved")

In [ ]:
# ── SECTION 12 : Save Model ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = [
    best_model_color if n == best_name else "steelblue"
    for n in comp_df.index
]
ax.bar(comp_df.index, comp_df["Test_R2"], color=bar_colors)
ax.set_title("Model Comparison — Test R²")
ax.set_ylabel("R²")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("plots/model_comparison.png", dpi=100, bbox_inches="tight")
plt.close()

os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/best_model.joblib")
joblib.dump(scaler,     "models/scaler.joblib")
print(f"  ✓ Best model '{best_name}' saved")

In [ ]:
# ── SECTION 13 : Final Summary ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("   FINAL RESULTS SUMMARY")
print("=" * 65)

# TODO: Print the winning model name, its Test R², Test RMSE, and Test MAE
#       from comp_df.  Use comp_df.index[0] for the best model name.
pass